In [1]:
import pandas as pd
import os
import requests

In [2]:
# 한국지능정보사회진흥원 디지털배움터 교육장 현황 api url
API_URL = 'https://api.odcloud.kr/api/15134509/v1/uddi:7172b895-8193-44c1-a3c7-c52322104653'
API_KEY = '0VGmzq6hyxxRhjYc0ZS1Fq2tHHoBnbBYMKCMbDKtnebtUS44dvI%2B0GuldwLIW3nk7d4rqcxFsPss9Fm6RQlWRQ%3D%3D'

In [3]:
def call_all_data():
    all_data = [] # 여기에 데이터 저장
    page = 1 # 페이지 총 2개인거 같은데 while로 총 데이터 개수까지 반복함
    per_page = 100

    while True:
        params = {
            'page': page,
            'perPage': per_page
        }

        headers = {
            'Authorization': API_KEY
        }

        response = requests.get(API_URL, params=params, headers=headers)
        response.raise_for_status() # 예외 자동 생성 
        result = response.json()

        data = result.get('data', [])
        if not data:
            break

        all_data.extend(data)

        if len(all_data) >= result.get('totalCount', 0):
            break

        page += 1

    return all_data

In [4]:
def create_csv():
    data = call_all_data() # 리스트 반환

    if not data:
        print("데이터가 없습니다.")
        return

    # DataFrame 생성
    df = pd.DataFrame(data)

    # 컬럼 순서 지정
    columns = ['관리지역', '배움터 유형', '배움터명', '배움터주소', '시군구', '이용정원']
    df = df[columns]

    # CSV 파일로 저장
    output_dir = '/Users/parkjuyong/Desktop/4-1/data-crawling-project/data'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'digital_learning_centers.csv')

    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"CSV 파일 생성 완료: {output_path}")
    print(f"총 {len(df)}개의 데이터 저장됨")

    return output_path